In [1]:
import torch
from torch import nn
from torch.utils.tensorboard import SummaryWriter

In [2]:
writer = SummaryWriter()

In [3]:
import os
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

NUM_WORKERS = os.cpu_count()

def createDataLoaders(train_dir:str, test_dir: str, transform: transforms.Compose, batch_size:int, num_workers: int = NUM_WORKERS):
    train_dataset = datasets.ImageFolder(root = train_dir, transform=transform, target_transform=None)
    test_dataset = datasets.ImageFolder(root = test_dir, transform=transform)

    classNames = train_dataset.classes

    train_dataloader = DataLoader(dataset= train_dataset, batch_size=batch_size, num_workers=num_workers, shuffle=True)
    test_dataloader = DataLoader(dataset= test_dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False)

    return train_dataloader, test_dataloader, classNames


In [4]:
class DesertClassifier(nn.Module):

    def __init__(self, input_shape: int, hidden_units: int, output_shape: int) ->None:
        super().__init__()

        self.firstConvBlock = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.secondConvBlock = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features = hidden_units * 16 * 16, out_features=output_shape)
        )

    def forward(self, x):
        return self.classifier(self.secondConvBlock(self.firstConvBlock(x)))


In [10]:
def trainStep(
        model: torch.nn.Module, 
        dataloader: torch.utils.data.DataLoader, 
        loss_fn: torch.nn.Module, optimizer: torch.optim.Optimizer
    ):

    model.train()

    train_loss = 0
    train_acc = 0

    for batch, (X, y) in enumerate(dataloader):

        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        predictedClass = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (predictedClass == y).sum().item() / len(y_pred)

    train_loss /= len(dataloader)
    train_acc /= len(dataloader)

    return train_loss, train_acc

def testStep(
        model: torch.nn.Module, 
        dataloader: torch.utils.data.DataLoader, 
        loss_fn: torch.nn.Module,
    ):

    model.eval()
    test_loss = 0
    test_acc = 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            y_pred = model(X)
            loss = loss_fn(y_pred, y)
            test_loss += loss.item()

            testPredictedLabels = y_pred.argmax(dim=1)
            test_acc += (testPredictedLabels == y).sum().item() / len(y_pred)

    test_loss /= len(dataloader)
    test_acc /= len(dataloader)

    return test_loss, test_acc

def train(
        model: torch.nn.Module, 
        train_dataloader: torch.utils.data.DataLoader, test_dataloader: torch.utils.data.DataLoader, 
        optimizer: torch.optim.Optimizer, loss_fn: torch.nn.Module = nn.CrossEntropyLoss(), epochs: int = 10,
        experimentName = "experiment"
    ):

    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    for epoch in range(epochs):
        train_loss, train_acc = trainStep(model= model, dataloader=train_dataloader, loss_fn=loss_fn, optimizer=optimizer)
        test_loss, test_acc = trainStep(model= model, dataloader=test_dataloader, loss_fn=loss_fn, optimizer=optimizer)

        print(f"Epoch: {epoch + 1}, Train loss: {train_loss}, Test loss: {test_loss}, Train acc: {train_acc}, Test acc: {test_acc}")

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        
        writer = SummaryWriter(log_dir=f"runs/{experimentName}")
        writer.add_scalars(main_tag="Loss", 
                           tag_scalar_dict={
                               "train loss" : train_loss,
                               "test loss": test_loss
                            },
                            global_step=epoch
        )

        writer.add_scalars(main_tag="Accuracy", 
                           tag_scalar_dict={
                               "train accuracy" : train_acc,
                               "test accuracy": test_acc
                            },
                            global_step=epoch
        )

        writer.add_graph(model=model, input_to_model=torch.randn(32, 3, 64, 64))

    writer.close()
    return results

In [11]:
NUM_EPOCHS = 5
BATCH_SIZE = 32
HIDDEN_UNITS = 32
LEARNING_RATE = 0.001

train_dir = "../data/desert101/train"
test_dir = "../data/desert101/test"

dataTransform = transforms.Compose(
    [
        transforms.Resize((64, 64)),
        transforms.RandomHorizontalFlip(p=0.4),
        transforms.TrivialAugmentWide(),
        transforms.ToTensor(),
        transforms.Normalize(mean = [0.5482, 0.4636, 0.3866], std = [0.2540, 0.2574, 0.2628])

    ]
)

train_dataloader, test_dataloader, classnames = createDataLoaders(train_dir=train_dir, test_dir=test_dir, transform=dataTransform, batch_size=BATCH_SIZE)

model = DesertClassifier(input_shape=3, hidden_units=HIDDEN_UNITS, output_shape=len(classnames))

lessNeuronModel = DesertClassifier(input_shape=3, hidden_units=10, output_shape=len(classnames))

In [ ]:
loss_fn = nn.CrossEntropyLoss()
train(model=model, 
      train_dataloader=train_dataloader, test_dataloader=test_dataloader, 
      loss_fn=loss_fn, optimizer=torch.optim.Adam(model.parameters(), lr=LEARNING_RATE),
      epochs=NUM_EPOCHS,
      experimentName="hidden_unit_32"
      )

train(model=lessNeuronModel, 
      train_dataloader=train_dataloader, test_dataloader=test_dataloader, 
      loss_fn=loss_fn, optimizer=torch.optim.Adam(lessNeuronModel.parameters(), lr=LEARNING_RATE),
      epochs=NUM_EPOCHS,
      experimentName="hidden_unit_32"
      )

Epoch: 1, Train loss: 1.4109662652015686, Test loss: 1.3842496077219646, Train acc: 0.25892857142857145, Test acc: 0.2676282051282051
Epoch: 2, Train loss: 1.366577684879303, Test loss: 1.3658097585042317, Train acc: 0.30669642857142854, Test acc: 0.2948717948717949
Epoch: 3, Train loss: 1.3247519731521606, Test loss: 1.334731658299764, Train acc: 0.340625, Test acc: 0.30128205128205127
Epoch: 4, Train loss: 1.284156572818756, Test loss: 1.3097552061080933, Train acc: 0.3834821428571428, Test acc: 0.4399038461538461
Epoch: 5, Train loss: 1.2501229882240295, Test loss: 1.24866517384847, Train acc: 0.42232142857142857, Test acc: 0.4302884615384615
Epoch: 1, Train loss: 1.3885400533676147, Test loss: 1.388593077659607, Train acc: 0.24776785714285715, Test acc: 0.20272435897435895
Epoch: 2, Train loss: 1.3817241311073303, Test loss: 1.370859185854594, Train acc: 0.2852678571428572, Test acc: 0.3004807692307692
Epoch: 3, Train loss: 1.363159728050232, Test loss: 1.3267625172932942, Train ac

{'train_loss': [1.3885400533676147,
  1.3817241311073303,
  1.363159728050232,
  1.3401570200920105,
  1.308874249458313],
 'train_acc': [0.24776785714285715,
  0.2852678571428572,
  0.29375,
  0.3169642857142857,
  0.40044642857142854],
 'test_loss': [1.388593077659607,
  1.370859185854594,
  1.3267625172932942,
  1.3255979617436726,
  1.2934366861979167],
 'test_acc': [0.20272435897435895,
  0.3004807692307692,
  0.4134615384615385,
  0.4703525641025641,
  0.4190705128205128]}

In [ ]:

!pip install --upgrade setuptools


  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0 requires setuptools<82, but you have setuptools 82.0.1 which is incompatible.


In [20]:
%load_ext tensorboard
%tensorboard --logdir runs # run session with the "runs/" directory (runs directory is created automatically)

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\1\Documents\GitHub\Deep Learning 2026\.venv\Scripts\tensorboard.exe\__main__.py", line 2, in <module>
    from tensorboard.main import run_main
  File "C:\Users\1\Documents\GitHub\Deep Learning 2026\.venv\Lib\site-packages\tensorboard\main.py", line 27, in <module>
    from tensorboard import default
  File "C:\Users\1\Documents\GitHub\Deep Learning 2026\.venv\Lib\site-packages\tensorboard\default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'